In [0]:
# ── Secrets from Key Vault ────────────────────────────────────────
storage_account = dbutils.secrets.get(scope="fraud-kv-scope", key="adls-storage-account-name")
storage_key     = dbutils.secrets.get(scope="fraud-kv-scope", key="adls-storage-account-key")
evh_conn_str    = dbutils.secrets.get(scope="fraud-kv-scope", key="evh-connection-string")
evh_namespace   = "evhb-fraud-pipeline"
evh_name        = "rt_transaction"


In [0]:
storage_account

'[REDACTED]'

In [0]:
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

In [0]:
# ── Paths ─────────────────────────────────────────────────────────
bronze_path      = f"abfss://bronze@{storage_account}.dfs.core.windows.net/transactions"
bronze_checkpoint = f"abfss://bronze@{storage_account}.dfs.core.windows.net/_checkpoints/transactions"

print("✅ Config loaded")

✅ Config loaded


In [0]:
import json

evh_conn_str = dbutils.secrets.get(scope="fraud-kv-scope", key="evh-connection-string")

# Native Event Hub config
eh_conf = {
    "eventhubs.connectionString": sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(evh_conn_str),
    "eventhubs.startingPosition": json.dumps({
        "offset": "-1",
        "seqNo": -1,
        "enqueuedTime": None,
        "isInclusive": True
    })
}

print("✅ Event Hub config ready")

✅ Event Hub config ready


In [0]:
raw_stream = (
    spark.readStream
    .format("eventhubs")
    .options(**eh_conf)
    .load()
)

print("✅ Stream reader created")
raw_stream.printSchema()

✅ Stream reader created
root
 |-- body: binary (nullable = true)
 |-- partition: string (nullable = true)
 |-- offset: string (nullable = true)
 |-- sequenceNumber: long (nullable = true)
 |-- enqueuedTime: timestamp (nullable = true)
 |-- publisher: string (nullable = true)
 |-- partitionKey: string (nullable = true)
 |-- properties: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- systemProperties: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



In [0]:
bronze_stream = (
    raw_stream
    .selectExpr(
        "CAST(body AS STRING)         AS raw_json",
        "CAST(partitionKey AS STRING) AS message_key",
        "partition",                             
        "offset",                                   
        "sequenceNumber               AS seq_number",
        "enqueuedTime                 AS event_hub_timestamp"
    )
)

query = (
    bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", bronze_checkpoint)
    .option("mergeSchema", "true")
    .trigger(processingTime="10 seconds")
    .start(bronze_path)
)

print(f"✅ Bronze stream started")
print(f"Status: {query.status}")

✅ Bronze stream started
Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [0]:
import time

# Monitor for 30 seconds then stop checking (stream keeps running)
for i in range(6):
    print(f"\n── Status check {i+1} ──")
    print(query.status)
    print(f"Recent progress: {query.recentProgress[-1] if query.recentProgress else 'No progress yet'}")
    time.sleep(5)


── Status check 1 ──
{'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
Recent progress: No progress yet

── Status check 2 ──
{'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Recent progress: No progress yet

── Status check 3 ──
{'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Recent progress: No progress yet

── Status check 4 ──
{'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Recent progress: No progress yet

── Status check 5 ──
{'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Recent progress: No progress yet

── Status check 6 ──
{'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Recent progress: No progress yet


In [0]:
bronze_df = spark.read.format('delta').load(bronze_path)
display(bronze_df)
print("count =>", bronze_df.count())

raw_json,message_key,partition,offset,seq_number,event_hub_timestamp
"{""transaction_id"": ""001657e2-c79b-4fd0-9f2f-29aa9cc75250"", ""customer_id"": ""CUST0352"", ""customer_name"": ""Kabir Bhat"", ""account_age_days"": 2085, ""avg_spend"": 933.94, ""spend_std"": 879.45, ""amount"": 2610.17, ""merchant"": ""BigBazaar"", ""location"": ""Mumbai"", ""is_foreign"": false, ""txn_type"": ""POS"", ""timestamp"": ""2026-04-03T07:26:07.091938""}",CUST0352,1,0,0,2026-04-03T07:26:07.15Z
"{""transaction_id"": ""17e6ae1e-d816-4d36-a427-86931a614922"", ""customer_id"": ""CUST0235"", ""customer_name"": ""Gokul Sood"", ""account_age_days"": 3193, ""avg_spend"": 5166.39, ""spend_std"": 3013.62, ""amount"": 3479.44, ""merchant"": ""Ola"", ""location"": ""Coimbatore"", ""is_foreign"": false, ""txn_type"": ""NEFT"", ""timestamp"": ""2026-04-03T07:26:08.332218""}",CUST0235,1,480,1,2026-04-03T07:26:07.353Z
"{""transaction_id"": ""4408a76f-22fc-44b9-8955-21bb918777e8"", ""customer_id"": ""CUST0339"", ""customer_name"": ""Mahika Mannan"", ""account_age_days"": 2401, ""avg_spend"": 2182.49, ""spend_std"": 3174.81, ""amount"": 5224.65, ""merchant"": ""Blinkit"", ""location"": ""Mumbai"", ""is_foreign"": false, ""txn_type"": ""online"", ""timestamp"": ""2026-04-03T07:26:08.833637""}",CUST0339,1,960,2,2026-04-03T07:26:07.853Z
"{""transaction_id"": ""e614f153-a997-4f45-9d7b-3297a5096d97"", ""customer_id"": ""CUST0373"", ""customer_name"": ""Nirvi Shere"", ""account_age_days"": 2231, ""avg_spend"": 2107.72, ""spend_std"": 2805.89, ""amount"": 13909.24, ""merchant"": ""UNKNOWN"", ""location"": ""Cayman Islands"", ""is_foreign"": true, ""txn_type"": ""UPI"", ""timestamp"": ""2026-04-03T01:26:09.335878""}",CUST0373,1,1448,3,2026-04-03T07:26:08.353Z
"{""transaction_id"": ""6ac669f4-cda0-4fef-a52d-9b12615fc79f"", ""customer_id"": ""CUST0371"", ""customer_name"": ""Adah Ahluwalia"", ""account_age_days"": 3233, ""avg_spend"": 6390.95, ""spend_std"": 8956.62, ""amount"": 6564.1, ""merchant"": ""Flipkart"", ""location"": ""Chennai"", ""is_foreign"": false, ""txn_type"": ""online"", ""timestamp"": ""2026-04-03T07:26:09.838745""}",CUST0371,1,1936,4,2026-04-03T07:26:08.853Z
"{""transaction_id"": ""46b195bf-8eba-4c95-834b-288648a68754"", ""customer_id"": ""CUST0214"", ""customer_name"": ""Ela Balay"", ""account_age_days"": 2670, ""avg_spend"": 11209.6, ""spend_std"": 8735.02, ""amount"": 17503.22, ""merchant"": ""DMart"", ""location"": ""Pune"", ""is_foreign"": false, ""txn_type"": ""UPI"", ""timestamp"": ""2026-04-03T07:26:10.340889""}",CUST0214,1,2424,5,2026-04-03T07:26:09.353Z
"{""transaction_id"": ""c077a082-5dce-47fa-b50f-6258b1839d98"", ""customer_id"": ""CUST0210"", ""customer_name"": ""Amani Dhawan"", ""account_age_days"": 2051, ""avg_spend"": 12212.56, ""spend_std"": 14213.5, ""amount"": 2552.12, ""merchant"": ""BookMyShow"", ""location"": ""Kolkata"", ""is_foreign"": false, ""txn_type"": ""ATM"", ""timestamp"": ""2026-04-03T07:26:10.842594""}",CUST0210,1,2904,6,2026-04-03T07:26:09.853Z
"{""transaction_id"": ""eee4c8d6-0bb7-4dc9-ae05-5e11eded206c"", ""customer_id"": ""CUST0118"", ""customer_name"": ""Amira Vig"", ""account_age_days"": 3539, ""avg_spend"": 11857.29, ""spend_std"": 10877.89, ""amount"": 10265.8, ""merchant"": ""Blinkit"", ""location"": ""Surat"", ""is_foreign"": false, ""txn_type"": ""NEFT"", ""timestamp"": ""2026-04-03T07:26:07.830222""}",CUST0118,0,0,0,2026-04-03T07:26:07.085Z


count => 8


In [0]:
for stream in spark.streams.active:
    print(f"Stopping: {stream.name} | {stream.id}")
    stream.stop()

print(f"Active streams: {len(spark.streams.active)}")

Active streams: 0
